In [ ]:
# === CABECALHO ACADEMICO (gerado por scripts/apply_header.py) ===
# Universidade Presbiteriana Mackenzie
# Disciplina: Inteligência Artificial
# Projeto: Previsão de desempenho de jogadores da NBA (pontos por jogo)
#
# INTEGRANTES DO GRUPO:
#   - Rodrigo Lucas Mascarenhas Leite Oliveira - TIA 10427925
#   - André Ihsan Ward - TIA 10425684
#
# ARQUIVO: notebooks/04_incerteza_teto_e_erros.ipynb
# SINTESE: Intervalos, teto preditivo, ablação e análise de erros.
#
# HISTORICO DE ALTERACOES:
#   data       | autor      | descricao
#   -----------|------------|-------------------------------------------
#   2026-11-15 | Rodrigo    | Versão inicial.
# === FIM DO CABECALHO ===


# 04 - Incerteza, teto preditivo, ablação e análise de erros

Compromissos éticos da N1 tratados aqui: (i) acompanhar cada estimativa de uma medida de dispersão; (ii) declarar o teto de desempenho; (iii) reportar métricas por faixa de minutos e por perfil de jogador.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from IPython.display import Image, display
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.max_columns", 30)
TV = json.loads(Path("results/metrics/treino_validacao.json").read_text(encoding="utf-8"))
TF = json.loads(Path("results/metrics/test_final.json").read_text(encoding="utf-8"))

## 1. Intervalos conformais

Intervalo `μ ± q·√max(μ, 1)`, limitado a zero: o modelo final prevê μ, e `q` é o quantil do erro normalizado na validação 2024-25 (predição conformal dividida). A cobertura garantida é marginal, não por subgrupo. A raiz de μ faz o intervalo crescer com a previsão, porque quem marca mais oscila mais.

In [ ]:
iv = TF["regressao"]["intervalos"]
pd.DataFrame({k: {"nominal": v["nominal"], "q": v["q"], **v["global"]} for k, v in iv.items()}).T

In [ ]:
pd.concat({k: pd.DataFrame(v["por_faixa_de_minutos"]).set_index("grupo")[["n", "cobertura", "largura_media"]] for k, v in iv.items()}, axis=1)

In [ ]:
display(Image("results/figures/f_cobertura_intervalos.png", width=640))

## 2. Teto preditivo

Previsores que **trapaceiam** de propósito, com informação do futuro: oráculos simples (média ou mediana do jogador na temporada inteira ou nos 5 jogos antes e depois) e o **modelo com nível oráculo** (o modelo final recebendo também a média real do jogador na temporada). Não são utilizáveis; servem de régua para medir quanto do erro se deve a não conhecer o nível do atleta.

In [ ]:
teto = TF["regressao"]["teto_preditivo"]
fin = TF["regressao"]["modelo_final"]
rows = {k: {"MAE": v["mae"], "modelo - oráculo": v["modelo_menos_oraculo"]["delta"], "IC inf": v["modelo_menos_oraculo"]["ic_inf"], "IC sup": v["modelo_menos_oraculo"]["ic_sup"]}
        for k, v in teto.items() if isinstance(v, dict)}
mo = TF["regressao"]["diagnosticos_posteriores"]["modelo_com_nivel_oraculo"]
rows["modelo final + nível oráculo"] = {"MAE": mo["mae_teste"], "modelo - oráculo": mo["modelo_final_menos_oraculo"]["delta"], "IC inf": mo["modelo_final_menos_oraculo"]["ic_inf"], "IC sup": mo["modelo_final_menos_oraculo"]["ic_sup"]}
rows["modelo final (" + fin + ")"] = {"MAE": TF["regressao"]["teste"][fin]["mae"]}
rows["média exponencial"] = {"MAE": TF["regressao"]["teste"]["media_exponencial"]["mae"]}
print(teto["nota"])
pd.DataFrame(rows).T

**Leitura:** os oráculos simples são estimadores ruidosos do nível (o local erra mais que a média exponencial) e não servem de limite superior. O modelo com nível oráculo é a régua útil: conhecer o nível verdadeiro do jogador reduziria o MAE em só ~0,07 ponto. O erro que sobra (~4,9 pontos por jogo) vem principalmente da variação de um jogo para outro (minutos, aproveitamento, contusões, ritmo). Não é um teto absoluto: um oráculo com outras informações (por exemplo, os minutos que o jogador vai jogar) iria mais longe.

## 3. Ablação por família (resultado esperado b)

In [ ]:
ab = TF["regressao"]["ablacao_por_familia"]
pd.DataFrame({k: {"MAE sem a família": v["mae_sem_familia"], **{kk: v["aumento_vs_completo"][kk] for kk in ("delta", "ic_inf", "ic_sup", "p_valor")}} for k, v in ab.items()}).T.sort_values("delta", ascending=False)

In [ ]:
display(Image("results/figures/f_ablacao_familias.png", width=640))
pi = TF["regressao"]["importancia_permutacao_agrupada"]
pd.DataFrame(pi["familias"]).T.sort_values("aumento_mae_medio", ascending=False)

O histórico de pontos domina. Adversário, minutos e descanso/mando têm contribuições pequenas; só a do adversário resiste a uma correção de Bonferroni para as seis comparações. O volume de jogo não tem efeito detectável na ablação (sua informação reaparece nas colunas do histórico), mas a permutação agrupada mostra que o modelo o usa.

## 4. Análise de erros

In [ ]:
perf = TF["regressao"]["por_perfil"]
pd.concat({grp: pd.concat({m: pd.DataFrame(v).set_index("grupo")[["n", "mae", "mae_relativo", "vies"]] for m, v in d.items()}, axis=1) for grp, d in perf.items()})

In [ ]:
pd.DataFrame(TF["regressao"]["vies_por_decil_da_previsao"])

In [ ]:
pd.DataFrame(TF["regressao"]["maiores_erros"]).head(12)

Os maiores erros são jogos excepcionais (atuações de 50+ pontos ou saídas precoces): eventos reais, sem marca no histórico pré-jogo. O viés por decil mostra o modelo abaixo da média real nos decis baixos e médios, efeito de prever a mediana condicional (que minimiza o erro absoluto) numa distribuição assimétrica à direita, e ligeiramente acima nos dois decis superiores, o que indica que o modelo encolhe pouco as previsões dos maiores pontuadores.